# Data Processing

In [ ]:
# ==========================================
# IMPORTS & ENVIRONMENT SETUP
# ==========================================

import os
import shutil
import random
import torch
import torchaudio
import torchvision
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import kagglehub
import matplotlib.pyplot as plt

# Set random seeds for reproducibility
random.seed(42)
torch.manual_seed(42)

print("PyTorch version:", torch.__version__)
print("Torchaudio version:", torchaudio.__version__)

In [ ]:
# ==========================================
# DOWNLOAD & FILE-LEVEL SPLIT
# ==========================================

# 1. Download Dataset
path = kagglehub.dataset_download(
    "andradaolteanu/gtzan-dataset-music-genre-classification"
)
source_dir = os.path.join(path, "Data", "genres_original")
target_root = "gtzan_audio_split"
splits = ["train", "val", "test"]

# 2. Create target directories
if os.path.exists(target_root):
    shutil.rmtree(target_root)

genres = [
    d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))
]
for split in splits:
    for genre in genres:
        os.makedirs(os.path.join(target_root, split, genre), exist_ok=True)

# 3. Perform 80/10/10 File Split (NO SLICING YET)
print("Distributing full 30s audio files to prevent data leakage...")
stats = {split: 0 for split in splits}

for genre in genres:
    genre_path = os.path.join(source_dir, genre)
    files = [f for f in os.listdir(genre_path) if f.endswith(".wav")]

    # Remove the corrupted jazz file if it exists
    if "jazz.00054.wav" in files:
        files.remove("jazz.00054.wav")

    random.shuffle(files)
    n = len(files)

    # 80% Train, 10% Val, 10% Test
    train_idx = int(n * 0.8)
    val_idx = int(n * 0.9)

    file_splits = {
        "train": files[:train_idx],
        "val": files[train_idx:val_idx],
        "test": files[val_idx:],
    }

    for split, split_files in file_splits.items():
        for f in split_files:
            src = os.path.join(genre_path, f)
            dst = os.path.join(target_root, split, genre, f)
            shutil.copy(src, dst)
            stats[split] += 1

print("File split complete.")
for split, count in stats.items():
    print(f" -> {split.capitalize()} set: {count} full songs")

In [ ]:
# ==========================================
# DATASET CLASS (3s SLICES & MEL SPECTROGRAMS)
# ==========================================


class GTZAN_3Second_Dataset(Dataset):
    def __init__(self, root_dir, split, sample_rate=22050, duration=3, augment=False):
        self.root_dir = os.path.join(root_dir, split)
        self.sample_rate = sample_rate
        self.n_samples = sample_rate * duration  # 3 seconds = 66,150 frames
        self.augment = augment
        self.file_list = []

        genres = sorted(os.listdir(self.root_dir))
        self.label_to_idx = {genre: i for i, genre in enumerate(genres)}

        # Register every 3-second slice as a separate dataset item
        for genre in genres:
            genre_dir = os.path.join(self.root_dir, genre)
            for f in os.listdir(genre_dir):
                if f.endswith(".wav"):
                    full_path = os.path.join(genre_dir, f)
                    # 10 slices per 30-second track
                    for segment_idx in range(10):
                        self.file_list.append(
                            (full_path, self.label_to_idx[genre], segment_idx)
                        )

        # Define Audio Transforms
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate, n_fft=2048, hop_length=512, n_mels=128
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB(top_db=80)

        # Define Augmentations
        self.pitch_shift = torchaudio.transforms.PitchShift(sample_rate, n_steps=2)

    def add_white_noise(self, waveform, noise_level=0.005):
        noise = torch.randn_like(waveform) * noise_level
        return waveform + noise

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label, segment_idx = self.file_list[idx]

        # Load exact 3-second chunk
        offset = segment_idx * self.n_samples
        try:
            waveform, sr = torchaudio.load(
                path, frame_offset=offset, num_frames=self.n_samples
            )
            if waveform.numel() == 0 or waveform.shape[1] == 0:
                waveform = torch.zeros((1, self.n_samples))
                sr = self.sample_rate
        except Exception:
            waveform = torch.zeros((1, self.n_samples))
            sr = self.sample_rate

        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(
                orig_freq=sr, new_freq=self.sample_rate
            )
            waveform = resampler(waveform)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        if self.augment:
            if random.random() > 0.5:
                waveform = self.pitch_shift(waveform)
            if random.random() > 0.5:
                waveform = self.add_white_noise(waveform)

        # Convert raw audio to Mel Spectrogram
        mel_spec = self.mel_spectrogram(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)

        # --- THE ULTIMATE FIX ---
        # Strictly enforce the size of the output image (spectrogram).
        # 130 frames is the target width for 3s of audio with a 512 hop_length.
        target_frames = 130

        # If it's too narrow, pad it with zeros (silence)
        if mel_spec_db.shape[2] < target_frames:
            mel_spec_db = F.pad(mel_spec_db, (0, target_frames - mel_spec_db.shape[2]))
        # If it's too wide, truncate the extra frames
        else:
            mel_spec_db = mel_spec_db[:, :, :target_frames]

        return mel_spec_db, torch.tensor(label)

In [ ]:
# ==========================================
# INITIALIZE DATALOADERS
# ==========================================

# Initialize Datasets (Notice augment=True ONLY for the training set)
train_dataset = GTZAN_3Second_Dataset(
    root_dir="gtzan_audio_split", split="train", augment=True
)
val_dataset = GTZAN_3Second_Dataset(
    root_dir="gtzan_audio_split", split="val", augment=False
)
test_dataset = GTZAN_3Second_Dataset(
    root_dir="gtzan_audio_split", split="test", augment=False
)

# Initialize DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Verification
mel_specs, labels = next(iter(train_loader))

print(f"Total Training Samples (3s Slices): {len(train_dataset)}")
print(f"Total Validation Samples: {len(val_dataset)}")
print(f"Total Test Samples: {len(test_dataset)}")
print(f"\\nBatch Shape: {mel_specs.shape} -> [Batch Size, Channels, Mels, Time Steps]")

# Plot a sample Mel Spectrogram to verify
plt.figure(figsize=(10, 4))
plt.imshow(
    mel_specs[0].squeeze().detach().numpy(), cmap="magma", origin="lower", aspect="auto"
)

plt.title(f"Sample Mel Spectrogram (Genre Label: {labels[0].item()})")
plt.ylabel("Mel Frequency Bins")
plt.xlabel("Time Frames")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

# Model Architecture

In [ ]:
# ==========================================
# THE SPECTROGRAM CNN
# ==========================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class SEBlock(nn.Module):
    """Squeeze-and-Excitation Attention Block: Learns to weight important feature channels."""

    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        # Squeeze: Get global context of each channel
        y = self.squeeze(x).view(b, c)
        # Excitation: Calculate attention weights
        y = self.excitation(y).view(b, c, 1, 1)
        # Scale: Multiply input by attention weights
        return x * y.expand_as(x)


class MultiScaleSpectroBlock(nn.Module):
    """Custom block using parallel asymmetric convolutions for Time and Frequency."""

    def __init__(self, in_channels, out_channels):
        super(MultiScaleSpectroBlock, self).__init__()

        # Stream 1: Standard spatial features
        self.conv_standard = nn.Conv2d(
            in_channels, out_channels // 3, kernel_size=3, padding=1
        )

        # Stream 2: Temporal features (Long width, narrow height) - Captures Rhythm
        self.conv_temporal = nn.Conv2d(
            in_channels, out_channels // 3, kernel_size=(1, 7), padding=(0, 3)
        )

        # Stream 3: Frequency features (Narrow width, tall height) - Captures Harmony/Pitch
        self.conv_frequency = nn.Conv2d(
            in_channels,
            out_channels - 2 * (out_channels // 3),
            kernel_size=(7, 1),
            padding=(3, 0),
        )

        self.bn = nn.BatchNorm2d(out_channels)
        self.se_block = SEBlock(out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # Run parallel convolutions
        out_std = self.conv_standard(x)
        out_tmp = self.conv_temporal(x)
        out_frq = self.conv_frequency(x)

        # Concatenate features along the channel dimension
        out = torch.cat([out_std, out_tmp, out_frq], dim=1)

        # Apply Batch Norm, Activation, Attention, and Pooling
        out = self.bn(out)
        out = F.leaky_relu(out, negative_slope=0.01)
        out = self.se_block(out)
        out = self.pool(out)

        return out


class SpectroCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SpectroCNN, self).__init__()

        # Initial Convolution to expand 1 grayscale channel into 32 feature maps
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.01),
        )

        # Stack of our custom Multi-Scale blocks
        self.block1 = MultiScaleSpectroBlock(32, 64)
        self.block2 = MultiScaleSpectroBlock(64, 128)
        self.block3 = MultiScaleSpectroBlock(128, 256)
        self.block4 = MultiScaleSpectroBlock(256, 512)

        # Global Average Pooling flattens the spatial dimensions robustly
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Advanced Classifier with Dropout
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.5),  # High dropout to prevent overfitting on complex features
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.init_conv(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.classifier(x)
        return x


# Initialize Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SpectroCNN(num_classes=10).to(device)

print(f"Model initialized on: {device}")
# Calculate total parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")

In [ ]:
# ==========================================
# TRAINING LOOP
# ==========================================

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# Training hyperparameters optimized for A100/V100
EPOCHS = 40
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
# AdamW provides better weight decay than standard Adam
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
# Slowly reduces learning rate to 0 by the last epoch
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses, val_accuracies = [], []
best_val_acc = 0.0

print("Starting Advanced Training Phase...")
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()

        # Gradient clipping prevents "exploding gradients" in deep networks
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # --- Validation Phase ---
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    val_accuracies.append(val_acc)

    # Step the learning rate scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # Save the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_spectro_cnn.pth")
        checkpoint_msg = "--> Best Model Saved!"
    else:
        checkpoint_msg = ""

    print(
        f"Epoch [{epoch+1:02d}/{EPOCHS}] | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.2f}% {checkpoint_msg}"
    )

total_time = (time.time() - start_time) / 60
print(f"\nTraining Completed in {total_time:.2f} minutes!")
print(f"Absolute Best Validation Accuracy: {best_val_acc:.2f}%")

In [ ]:
# ==========================================
# TEST SET EVALUATION & GRAPHICS
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import os

# 1. Load the absolute best weights saved during training
# (This ensures we don't just use the weights from the final epoch if it overfit slightly at the end)
model.load_state_dict(torch.load("best_spectro_cnn.pth"))
model.eval()

y_true = []
y_pred = []

print("Running inference on the strictly isolated Test Set...")

# Disable gradient tracking for faster inference
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# Get the ordered list of genres for our labels
genres = sorted(os.listdir("gtzan_audio_split/test"))

# ==========================================
# VISUALIZATION 1: CLASSIFICATION REPORT (TABLE)
# ==========================================
print("\n" + "=" * 50)
print(" FINAL CLASSIFICATION REPORT (TEST SET)")
print("=" * 50)
# This automatically formats Precision, Recall, and F1 into a clean table
print(classification_report(y_true, y_pred, target_names=genres))

# ==========================================
# VISUALIZATION 2: CONFUSION MATRIX (HEATMAP)
# ==========================================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 9))
# Using a custom color map (mako) that looks incredibly clean and professional
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="mako_r",
    xticklabels=genres,
    yticklabels=genres,
    linewidths=0.5,
    cbar_kws={"shrink": 0.75},
)

plt.title("Advanced Spectrogram CNN - Final Confusion Matrix", fontsize=16, pad=20)
plt.ylabel("Actual True Genre", fontsize=12, labelpad=10)
plt.xlabel("Model Predicted Genre", fontsize=12, labelpad=10)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()

print("\nGenerating Confusion Matrix Graphic...")
plt.show()